In [ ]:
# =============================================================================
# 12_create_biweekly_satellite_data.ipynb
#
# PURPOSE
# =============================================================================
#
# Create 14-day Sentinel-1 and Sentinel-2 composite imagery from the
# acquisition-level satellite data created by Notebook 10.
#
#
# SOURCE DATA
# =============================================================================
#
# finals/
# └── daily_datasets/
#
#     ├── sentinel1/
#     ├── sentinel2/
#     └── daily_satellite_inventory.csv
#
#
# Notebook 12 DOES NOT:
#
#   - connect to Earth Engine
#   - download satellite data
#   - modify acquisition-level imagery
#   - create another daily_datasets folder
#
#
# Notebook 10 is the single source of truth for acquisition-level imagery.
#
#
# TEMPORAL DESIGN
# =============================================================================
#
# BEFORE:
#
#   2024-05-10 through 2024-09-26
#
#   140 calendar days
#   exactly 10 × 14-day periods
#
#
# AFTER:
#
#   2024-09-27 through 2025-02-13
#
#   140 calendar days
#   exactly 10 × 14-day periods
#
#
# Therefore every site has a fixed potential panel:
#
#   before_P01 ... before_P10
#
#   after_P01  ... after_P10
#
#
# September 27 is the first AFTER date.
#
#
# COMPOSITING RULE
# =============================================================================
#
# For every:
#
#   site × sensor × period
#
#
# 0 acquisitions:
#
#   - no TIFF is created
#   - the period remains in the quality table as missing
#
#
# 1 acquisition:
#
#   - biweekly image equals the single acquisition
#
#
# 2+ acquisitions:
#
#   - pixel-wise NaN-aware median
#
#
# Example:
#
# Acquisition 1:
#
#   pixel = NaN
#
# Acquisition 2:
#
#   pixel = 0.52
#
# Acquisition 3:
#
#   pixel = 0.61
#
# Biweekly composite:
#
#   pixel = median(0.52, 0.61)
#
#
# PRIMARY QUALITY DEFINITION
# =============================================================================
#
# A spatial pixel is VALID when AT LEAST ONE output band has a finite value.
#
# Primary:
#
#   valid_pixel_fraction
#   valid_pixel_percentage
#
#
# Also retained:
#
#   valid_pixel_fraction_any_band
#   valid_pixel_fraction_all_bands
#
#
# RESUME-SAFE BEHAVIOR
# =============================================================================
#
# Before creating a biweekly image:
#
#   1. Check whether the expected TIFF already exists.
#
#   2. Existing + valid:
#
#        DO NOT rebuild.
#
#        Read existing TIFF.
#        Recalculate quality.
#
#        build_action = skipped_existing
#
#
#   3. Existing + invalid:
#
#        rename to *.invalid
#        recreate from daily TIFFs
#
#
#   4. Missing:
#
#        create from daily TIFFs
#
#
# OUTPUT STRUCTURE
# =============================================================================
#
# finals/
# └── biweekly_datasets/
#
#     ├── sentinel1/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   │
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── sentinel2/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   │
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── biweekly_image_quality.csv
#     ├── biweekly_image_quality.xlsx
#     ├── biweekly_quality_summary.csv
#     ├── biweekly_period_definitions.csv
#     ├── biweekly_period_dimension.csv
#     ├── biweekly_site_dimension.csv
#     ├── biweekly_build_action_summary.csv
#     └── biweekly_folder_summary.csv
#
# =============================================================================


# =============================================================================
# 1. Packages
# =============================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import rasterio


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)


print(
    "Packages loaded successfully."
)


# =============================================================================
# 2. Project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)


FINALS_DIR = (
    BASE_DIR /
    "finals"
)


# -----------------------------------------------------------------------------
# Notebook 10 source
# -----------------------------------------------------------------------------

DAILY_DIR = (
    FINALS_DIR /
    "daily_datasets"
)


DAILY_INVENTORY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


# -----------------------------------------------------------------------------
# Notebook 12 output
#
# IMPORTANT:
#
# There is NO finals/biweekly/ folder anymore.
#
# Everything goes directly under finals/biweekly_datasets/
# -----------------------------------------------------------------------------

BIWEEKLY_DIR = (
    FINALS_DIR /
    "biweekly_datasets"
)


S1_BIWEEKLY_DIR = (
    BIWEEKLY_DIR /
    "sentinel1"
)


S2_BIWEEKLY_DIR = (
    BIWEEKLY_DIR /
    "sentinel2"
)


QUALITY_CSV_FILE = (
    BIWEEKLY_DIR /
    "biweekly_image_quality.csv"
)


QUALITY_EXCEL_FILE = (
    BIWEEKLY_DIR /
    "biweekly_image_quality.xlsx"
)


SUMMARY_CSV_FILE = (
    BIWEEKLY_DIR /
    "biweekly_quality_summary.csv"
)


PERIOD_DEFINITION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_period_definitions.csv"
)


PERIOD_DIMENSION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_period_dimension.csv"
)


SITE_DIMENSION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_site_dimension.csv"
)


BUILD_ACTION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_build_action_summary.csv"
)


FOLDER_SUMMARY_FILE = (
    BIWEEKLY_DIR /
    "biweekly_folder_summary.csv"
)


BIWEEKLY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3. Check Notebook 10 output
# =============================================================================

if not DAILY_INVENTORY_FILE.exists():

    raise FileNotFoundError(
        "Notebook 10 inventory was not found:\n\n"
        f"{DAILY_INVENTORY_FILE}\n\n"
        "Run Notebook 10 first."
    )


print(
    "\nNotebook 10 acquisition inventory:"
)


print(
    DAILY_INVENTORY_FILE
)


# =============================================================================
# 4. Study period
# =============================================================================

HELENE_REFERENCE_DATE = pd.Timestamp(
    "2024-09-27"
)


BEFORE_START = pd.Timestamp(
    "2024-05-10"
)


BEFORE_END = pd.Timestamp(
    "2024-09-26"
)


AFTER_START = pd.Timestamp(
    "2024-09-27"
)


AFTER_END = pd.Timestamp(
    "2025-02-13"
)


STUDY_START = (
    BEFORE_START
)


STUDY_END = (
    AFTER_END
)


print(
    "\nStudy period:"
)


print(
    STUDY_START.strftime(
        "%Y-%m-%d"
    ),
    "through",
    STUDY_END.strftime(
        "%Y-%m-%d"
    ),
)


print(
    "\nBefore:"
)


print(
    BEFORE_START.strftime(
        "%Y-%m-%d"
    ),
    "through",
    BEFORE_END.strftime(
        "%Y-%m-%d"
    ),
)


print(
    "\nAfter:"
)


print(
    AFTER_START.strftime(
        "%Y-%m-%d"
    ),
    "through",
    AFTER_END.strftime(
        "%Y-%m-%d"
    ),
)


# =============================================================================
# 5. Settings
# =============================================================================

# Reuse valid biweekly TIFFs.
SKIP_EXISTING_BIWEEKLY = True


# False = process all treatment and counterfactual sites
# available in Notebook 10.
RUN_TEST_ONLY = False


# =============================================================================
# 6. Sensor bands
# =============================================================================

SENSOR_BANDS = {

    "sentinel1": [

        "VV",

        "VH",

        "VV_minus_VH",

    ],


    "sentinel2": [

        "B2",

        "B3",

        "B4",

        "B8",

        "B11",

        "B12",

        "NDVI",

        "NDWI",

    ],

}


# =============================================================================
# 7. Create output folders
# =============================================================================

for sensor_root in [

    S1_BIWEEKLY_DIR,

    S2_BIWEEKLY_DIR,

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (
                sensor_root /
                group /
                period
            )


            folder.mkdir(
                parents=True,
                exist_ok=True,
            )


print(
    "\nBiweekly output directory:"
)


print(
    BIWEEKLY_DIR
)


# =============================================================================
# 8. Generate exactly 10 × 14-day periods on each side
# =============================================================================

def create_14day_periods(
    period_name,
    start_date,
    number_of_periods=10,
):

    records = []


    for period_number in range(
        1,
        number_of_periods + 1,
    ):

        period_start = (
            start_date
            +
            pd.Timedelta(
                days=
                    (
                        period_number - 1
                    )
                    *
                    14
            )
        )


        period_end = (
            period_start
            +
            pd.Timedelta(
                days=13
            )
        )


        records.append(
            {

                "period":
                    period_name,

                "period_number":
                    period_number,

                "period_id":
                    (
                        f"{period_name}_"
                        f"P{period_number:02d}"
                    ),

                "period_start":
                    period_start,

                "period_end":
                    period_end,

                "calendar_days":
                    14,

            }
        )


    return records


before_periods = (
    create_14day_periods(
        period_name=
            "before",

        start_date=
            BEFORE_START,
    )
)


after_periods = (
    create_14day_periods(
        period_name=
            "after",

        start_date=
            AFTER_START,
    )
)


period_definitions = pd.DataFrame(
    before_periods
    +
    after_periods
)


# =============================================================================
# 9. Validate exact temporal structure
# =============================================================================

before_definition = (
    period_definitions
    .loc[
        period_definitions[
            "period"
        ]
        ==
        "before"
    ]
)


after_definition = (
    period_definitions
    .loc[
        period_definitions[
            "period"
        ]
        ==
        "after"
    ]
)


assert (
    len(
        before_definition
    )
    ==
    10
)


assert (
    len(
        after_definition
    )
    ==
    10
)


assert (
    before_definition.iloc[
        0
    ][
        "period_start"
    ]
    ==
    BEFORE_START
)


assert (
    before_definition.iloc[
        -1
    ][
        "period_end"
    ]
    ==
    BEFORE_END
)


assert (
    after_definition.iloc[
        0
    ][
        "period_start"
    ]
    ==
    AFTER_START
)


assert (
    after_definition.iloc[
        -1
    ][
        "period_end"
    ]
    ==
    AFTER_END
)


period_definitions.to_csv(
    PERIOD_DEFINITION_FILE,
    index=False,
)


print(
    "\nBiweekly period definitions:"
)


print(
    period_definitions[
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 10. Load Notebook 10 inventory
# =============================================================================

inventory = pd.read_csv(
    DAILY_INVENTORY_FILE
)


required_columns = [

    "site_id",

    "group",

    "sensor",

    "acquisition_date",

    "file_path",

]


missing_columns = [

    column

    for column
    in required_columns

    if column not in inventory.columns

]


if missing_columns:

    raise ValueError(
        "Notebook 10 inventory is missing required columns:\n"
        +
        str(
            missing_columns
        )
    )


inventory[
    "site_id"
] = (
    inventory[
        "site_id"
    ]
    .astype(str)
)


inventory[
    "group"
] = (
    inventory[
        "group"
    ]
    .astype(str)
)


inventory[
    "sensor"
] = (
    inventory[
        "sensor"
    ]
    .astype(str)
)


inventory[
    "acquisition_date"
] = pd.to_datetime(
    inventory[
        "acquisition_date"
    ]
)


print(
    "\nRaw Notebook 10 inventory rows:"
)


print(
    len(
        inventory
    )
)


# =============================================================================
# 11. Keep successful/existing acquisition TIFFs
# =============================================================================

if "status" in inventory.columns:

    inventory = (
        inventory
        .loc[
            inventory[
                "status"
            ].isin(
                [
                    "success",
                    "existing",
                ]
            )
        ]
        .copy()
    )


# =============================================================================
# 12. Restrict to exact study dates
# =============================================================================

inventory = (
    inventory
    .loc[
        (
            inventory[
                "acquisition_date"
            ]
            >=
            STUDY_START
        )
        &
        (
            inventory[
                "acquisition_date"
            ]
            <=
            STUDY_END
        )
    ]
    .copy()
)


# =============================================================================
# 13. Check local acquisition TIFFs
# =============================================================================

inventory[
    "file_exists"
] = (
    inventory[
        "file_path"
    ]
    .astype(str)
    .apply(
        lambda path:
            Path(
                path
            ).exists()
    )
)


missing_source_files = (
    inventory
    .loc[
        ~inventory[
            "file_exists"
        ]
    ]
    .copy()
)


if not missing_source_files.empty:

    print(
        "\nWARNING:"
    )


    print(
        len(
            missing_source_files
        ),
        "Notebook 10 inventory rows point to missing TIFF files."
    )


    print(
        "These acquisitions will not be used."
    )


inventory = (
    inventory
    .loc[
        inventory[
            "file_exists"
        ]
    ]
    .copy()
)


print(
    "\nUsable acquisition-level images:"
)


print(
    len(
        inventory
    )
)


# =============================================================================
# 14. Inventory diagnostics
# =============================================================================

print(
    "\nSensors:"
)


print(
    inventory[
        "sensor"
    ]
    .value_counts()
)


print(
    "\nGroups:"
)


print(
    inventory[
        "group"
    ]
    .value_counts()
)


print(
    "\nUnique treatment sites:"
)


print(
    inventory
    .loc[
        inventory[
            "group"
        ]
        ==
        "treatment",
        "site_id",
    ]
    .nunique()
)


print(
    "\nUnique counterfactual sites:"
)


print(
    inventory
    .loc[
        inventory[
            "group"
        ]
        ==
        "counterfactual",
        "site_id",
    ]
    .nunique()
)


# =============================================================================
# 15. Read GeoTIFF
# =============================================================================

def read_raster(
    file_path,
):

    file_path = Path(
        file_path
    )


    with rasterio.open(
        file_path
    ) as src:

        data = (
            src
            .read(
                masked=True
            )
            .astype(
                "float32"
            )
            .filled(
                np.nan
            )
        )


        result = {

            "data":
                data,

            "profile":
                src.profile.copy(),

            "transform":
                src.transform,

            "crs":
                src.crs,

            "width":
                int(
                    src.width
                ),

            "height":
                int(
                    src.height
                ),

            "band_count":
                int(
                    src.count
                ),

        }


    return result


# =============================================================================
# 16. Inspect TIFF and calculate quality
# =============================================================================

def inspect_tiff(
    file_path,
    band_names,
):

    file_path = Path(
        file_path
    )


    if not file_path.exists():

        return {

            "reusable":
                False,

            "validation_status":
                "missing",

            "error":
                "File does not exist.",

        }


    try:

        raster = (
            read_raster(
                file_path
            )
        )


        if (
            raster[
                "band_count"
            ]
            !=
            len(
                band_names
            )
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "wrong_band_count",

                "error":
                    (
                        f"Expected {len(band_names)} bands, "
                        f"found {raster['band_count']}."
                    ),

            }


        if (
            raster[
                "width"
            ]
            <=
            0
            or
            raster[
                "height"
            ]
            <=
            0
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "invalid_dimensions",

                "error":
                    "Raster dimensions are invalid.",

            }


        if (
            raster[
                "crs"
            ]
            is None
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "missing_crs",

                "error":
                    "Raster has no CRS.",

            }


        data = (
            raster[
                "data"
            ]
        )


        finite = np.isfinite(
            data
        )


        # ---------------------------------------------------------------------
        # PRIMARY:
        #
        # at least one band valid
        # ---------------------------------------------------------------------

        valid_any = (
            finite
            .any(
                axis=0
            )
        )


        # ---------------------------------------------------------------------
        # Strict diagnostic:
        #
        # all bands valid
        # ---------------------------------------------------------------------

        valid_all = (
            finite
            .all(
                axis=0
            )
        )


        result = {

            "reusable":
                True,

            "validation_status":
                "valid",

            "error":
                None,

            # Primary
            "valid_pixel_fraction":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            # Explicit any-band
            "valid_pixel_fraction_any_band":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage_any_band":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            # Strict all-band
            "valid_pixel_fraction_all_bands":
                float(
                    valid_all.mean()
                ),

            "valid_pixel_percentage_all_bands":
                float(
                    valid_all.mean()
                    *
                    100
                ),

        }


        # ---------------------------------------------------------------------
        # Band-specific quality
        # ---------------------------------------------------------------------

        for band_index, band_name in enumerate(
            band_names
        ):

            fraction = float(
                finite[
                    band_index
                ].mean()
            )


            result[
                f"valid_fraction_{band_name}"
            ] = (
                fraction
            )


            result[
                f"valid_percentage_{band_name}"
            ] = (
                fraction
                *
                100
            )


        return result


    except Exception as error:

        return {

            "reusable":
                False,

            "validation_status":
                "corrupt_or_unreadable",

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 17. Raster compatibility
# =============================================================================

def check_raster_compatibility(
    raster_infos,
):

    if len(
        raster_infos
    ) <= 1:

        return (
            True,
            None,
        )


    reference = (
        raster_infos[
            0
        ]
    )


    for image_number, current in enumerate(
        raster_infos[
            1:
        ],
        start=2,
    ):

        # ---------------------------------------------------------------------
        # Shape
        # ---------------------------------------------------------------------

        if (
            current[
                "data"
            ].shape
            !=
            reference[
                "data"
            ].shape
        ):

            return (

                False,

                (
                    f"Image {image_number} shape mismatch: "
                    f"{current['data'].shape} vs "
                    f"{reference['data'].shape}"
                ),

            )


        # ---------------------------------------------------------------------
        # CRS
        # ---------------------------------------------------------------------

        if (
            str(
                current[
                    "crs"
                ]
            )
            !=
            str(
                reference[
                    "crs"
                ]
            )
        ):

            return (

                False,

                f"Image {image_number} CRS mismatch.",

            )


        # ---------------------------------------------------------------------
        # Grid
        # ---------------------------------------------------------------------

        if (
            current[
                "transform"
            ]
            !=
            reference[
                "transform"
            ]
        ):

            return (

                False,

                f"Image {image_number} pixel-grid mismatch.",

            )


    return (
        True,
        None,
    )


# =============================================================================
# 18. Create biweekly median
# =============================================================================

def create_biweekly_composite(
    source_files,
):

    raster_infos = [

        read_raster(
            file_path
        )

        for file_path
        in source_files

    ]


    compatible, error = (
        check_raster_compatibility(
            raster_infos
        )
    )


    if not compatible:

        raise ValueError(
            error
        )


    # -------------------------------------------------------------------------
    # One acquisition
    # -------------------------------------------------------------------------

    if len(
        raster_infos
    ) == 1:

        composite = (
            raster_infos[
                0
            ][
                "data"
            ]
            .copy()
        )


        return (
            composite,
            raster_infos[
                0
            ],
        )


    # -------------------------------------------------------------------------
    # Multiple acquisitions
    # -------------------------------------------------------------------------

    stack = np.stack(

        [

            raster[
                "data"
            ]

            for raster
            in raster_infos

        ],

        axis=0,

    )


    # Dimensions:
    #
    # acquisition × band × row × column

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )


        composite = (
            np.nanmedian(
                stack,
                axis=0,
            )
        )


    return (
        composite,
        raster_infos[
            0
        ],
    )


# =============================================================================
# 19. Save biweekly GeoTIFF
# =============================================================================

def save_biweekly_composite(
    composite,
    reference_info,
    output_file,
):

    output_file = Path(
        output_file
    )


    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    profile = (
        reference_info[
            "profile"
        ]
        .copy()
    )


    profile.update(
        {

            "driver":
                "GTiff",

            "dtype":
                "float32",

            "count":
                int(
                    composite.shape[
                        0
                    ]
                ),

            "compress":
                "deflate",

            "nodata":
                np.nan,

        }
    )


    with rasterio.open(
        output_file,
        "w",
        **profile,
    ) as dst:

        dst.write(
            composite.astype(
                "float32"
            )
        )


# =============================================================================
# 20. Invalid backup path
# =============================================================================

def get_invalid_backup_path(
    file_path,
):

    file_path = Path(
        file_path
    )


    candidate = (
        file_path
        .with_name(
            file_path.name
            +
            ".invalid"
        )
    )


    counter = 1


    while candidate.exists():

        candidate = (
            file_path
            .with_name(
                file_path.name
                +
                f".invalid_{counter}"
            )
        )


        counter += 1


    return candidate


# =============================================================================
# 21. Quality label
# =============================================================================

def quality_label(
    valid_fraction,
):

    if pd.isna(
        valid_fraction
    ):

        return "missing"


    if valid_fraction >= 0.80:

        return "excellent"


    if valid_fraction >= 0.50:

        return "usable"


    if valid_fraction >= 0.20:

        return "limited"


    if valid_fraction > 0:

        return "poor"


    return "unusable"


# =============================================================================
# 22. Identify Notebook 10 source-quality variable
# =============================================================================

def identify_source_quality_column(
    dataframe,
):

    candidates = [

        "valid_pixel_fraction",

        "valid_pixel_fraction_any_band",

        "valid_pixel_fraction_all_bands",

    ]


    for column in candidates:

        if column in dataframe.columns:

            return column


    return None


SOURCE_QUALITY_COLUMN = (
    identify_source_quality_column(
        inventory
    )
)


print(
    "\nSource-image quality variable:"
)


print(
    SOURCE_QUALITY_COLUMN
)


# =============================================================================
# 23. Build/reuse biweekly image
# =============================================================================

def build_or_reuse_biweekly(
    source_files,
    output_file,
    band_names,
):

    output_file = Path(
        output_file
    )


    # =========================================================================
    # Existing image
    # =========================================================================

    if (
        SKIP_EXISTING_BIWEEKLY
        and
        output_file.exists()
    ):

        inspection = (
            inspect_tiff(
                output_file,
                band_names,
            )
        )


        if inspection.get(
            "reusable",
            False
        ):

            print(
                "    Existing valid biweekly TIFF — skipping rebuild."
            )


            inspection[
                "build_action"
            ] = (
                "skipped_existing"
            )


            inspection[
                "composite_created"
            ] = 1


            return inspection


        # ---------------------------------------------------------------------
        # Existing but invalid
        # ---------------------------------------------------------------------

        print(
            "    Existing biweekly TIFF failed validation:"
        )


        print(
            "   ",
            inspection.get(
                "validation_status"
            ),
            "|",
            inspection.get(
                "error"
            ),
        )


        invalid_backup = (
            get_invalid_backup_path(
                output_file
            )
        )


        try:

            output_file.rename(
                invalid_backup
            )


            print(
                "    Invalid TIFF moved to:"
            )


            print(
                "   ",
                invalid_backup
            )


        except Exception:

            try:

                output_file.unlink()

            except Exception:

                pass


    # =========================================================================
    # Create new composite
    # =========================================================================

    try:

        composite, reference_info = (
            create_biweekly_composite(
                source_files
            )
        )


        save_biweekly_composite(

            composite=
                composite,

            reference_info=
                reference_info,

            output_file=
                output_file,

        )


        inspection = (
            inspect_tiff(
                output_file,
                band_names,
            )
        )


        if not inspection.get(
            "reusable",
            False
        ):

            return {

                "composite_created":
                    0,

                "build_action":
                    "failed",

                "valid_pixel_fraction":
                    np.nan,

                "valid_pixel_percentage":
                    np.nan,

                "error":
                    inspection.get(
                        "error"
                    ),

            }


        inspection[
            "composite_created"
        ] = 1


        inspection[
            "build_action"
        ] = (
            "created"
        )


        return inspection


    except Exception as error:

        return {

            "composite_created":
                0,

            "build_action":
                "failed",

            "valid_pixel_fraction":
                np.nan,

            "valid_pixel_percentage":
                np.nan,

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 24. Site × sensor master
# =============================================================================

site_sensor_combinations = (
    inventory[
        [
            "site_id",
            "group",
            "sensor",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "group",
            "site_id",
            "sensor",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nSite × sensor combinations:"
)


print(
    len(
        site_sensor_combinations
    )
)


# =============================================================================
# 25. Test/full mode
# =============================================================================

if RUN_TEST_ONLY:

    treatment_ids = (
        inventory
        .loc[
            inventory[
                "group"
            ]
            ==
            "treatment",
            "site_id",
        ]
        .drop_duplicates()
        .tolist()
    )


    counterfactual_ids = (
        inventory
        .loc[
            inventory[
                "group"
            ]
            ==
            "counterfactual",
            "site_id",
        ]
        .drop_duplicates()
        .tolist()
    )


    selected_ids = []


    if treatment_ids:

        selected_ids.append(
            treatment_ids[
                0
            ]
        )


    if counterfactual_ids:

        selected_ids.append(
            counterfactual_ids[
                0
            ]
        )


    site_sensor_to_process = (
        site_sensor_combinations
        .loc[
            site_sensor_combinations[
                "site_id"
            ]
            .isin(
                selected_ids
            )
        ]
        .copy()
    )


    print(
        "\nTEST MODE ACTIVE"
    )


    print(
        "Selected sites:",
        selected_ids
    )


else:

    site_sensor_to_process = (
        site_sensor_combinations
        .copy()
    )


    print(
        "\nFULL DATASET MODE"
    )


# =============================================================================
# 26. Generate biweekly composites
# =============================================================================

biweekly_records = []


total_combinations = len(
    site_sensor_to_process
)


for combination_number, (_, combination) in enumerate(
    site_sensor_to_process.iterrows(),
    start=1,
):

    site_id = str(
        combination[
            "site_id"
        ]
    )


    group = str(
        combination[
            "group"
        ]
    )


    sensor = str(
        combination[
            "sensor"
        ]
    )


    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{combination_number}/{total_combinations}"
    )


    print(
        site_id,
        "|",
        group,
        "|",
        sensor,
    )


    print(
        "=" * 100
    )


    site_inventory = (
        inventory
        .loc[
            (
                inventory[
                    "site_id"
                ]
                ==
                site_id
            )
            &
            (
                inventory[
                    "group"
                ]
                ==
                group
            )
            &
            (
                inventory[
                    "sensor"
                ]
                ==
                sensor
            )
        ]
        .copy()
    )


    # -------------------------------------------------------------------------
    # Output root
    # -------------------------------------------------------------------------

    if sensor == "sentinel1":

        sensor_root = (
            S1_BIWEEKLY_DIR
        )


    elif sensor == "sentinel2":

        sensor_root = (
            S2_BIWEEKLY_DIR
        )


    else:

        continue


    band_names = (
        SENSOR_BANDS[
            sensor
        ]
    )


    # =========================================================================
    # Iterate through all 20 fixed periods
    # =========================================================================

    for _, period_row in (
        period_definitions.iterrows()
    ):

        period = str(
            period_row[
                "period"
            ]
        )


        period_number = int(
            period_row[
                "period_number"
            ]
        )


        period_id = str(
            period_row[
                "period_id"
            ]
        )


        period_start = pd.Timestamp(
            period_row[
                "period_start"
            ]
        )


        period_end = pd.Timestamp(
            period_row[
                "period_end"
            ]
        )


        # ---------------------------------------------------------------------
        # Acquisition images in this 14-day window
        # ---------------------------------------------------------------------

        acquisitions = (
            site_inventory
            .loc[
                (
                    site_inventory[
                        "acquisition_date"
                    ]
                    >=
                    period_start
                )
                &
                (
                    site_inventory[
                        "acquisition_date"
                    ]
                    <=
                    period_end
                )
            ]
            .sort_values(
                "acquisition_date"
            )
            .copy()
        )


        acquisition_count = (
            len(
                acquisitions
            )
        )


        acquisition_dates = (
            acquisitions[
                "acquisition_date"
            ]
            .dt.strftime(
                "%Y-%m-%d"
            )
            .tolist()
        )


        source_files = (
            acquisitions[
                "file_path"
            ]
            .astype(str)
            .tolist()
        )


        # ---------------------------------------------------------------------
        # Source acquisition quality
        # ---------------------------------------------------------------------

        source_mean_valid_fraction = (
            np.nan
        )


        source_best_valid_fraction = (
            np.nan
        )


        if (
            acquisition_count
            >
            0
            and
            SOURCE_QUALITY_COLUMN
            is not None
        ):

            quality_values = (
                acquisitions[
                    SOURCE_QUALITY_COLUMN
                ]
                .dropna()
            )


            if len(
                quality_values
            ) > 0:

                source_mean_valid_fraction = float(
                    quality_values.mean()
                )


                source_best_valid_fraction = float(
                    quality_values.max()
                )


        # ---------------------------------------------------------------------
        # Base record
        # ---------------------------------------------------------------------

        record = {

            "site_id":
                site_id,

            "group":
                group,

            "sensor":
                sensor,

            "period":
                period,

            "period_number":
                period_number,

            "period_id":
                period_id,

            "period_start":
                period_start.strftime(
                    "%Y-%m-%d"
                ),

            "period_end":
                period_end.strftime(
                    "%Y-%m-%d"
                ),

            "calendar_days":
                14,

            "acquisition_count":
                acquisition_count,

            "acquisition_dates":
                json.dumps(
                    acquisition_dates
                ),

            "source_files":
                json.dumps(
                    source_files
                ),

            "has_data":
                int(
                    acquisition_count
                    >
                    0
                ),

            "multiple_acquisitions":
                int(
                    acquisition_count
                    >
                    1
                ),

            "source_mean_valid_pixel_fraction":
                source_mean_valid_fraction,

            "source_best_valid_pixel_fraction":
                source_best_valid_fraction,

            "composite_created":
                0,

            "build_action":
                "missing",

            "output_file":
                None,

            "valid_pixel_fraction":
                np.nan,

            "valid_pixel_percentage":
                np.nan,

            "valid_pixel_fraction_any_band":
                np.nan,

            "valid_pixel_percentage_any_band":
                np.nan,

            "valid_pixel_fraction_all_bands":
                np.nan,

            "valid_pixel_percentage_all_bands":
                np.nan,

            "quality_label":
                "missing",

            "improvement_vs_mean_source":
                np.nan,

            "improvement_vs_best_source":
                np.nan,

            "error":
                None,

        }


        print(
            "\n",
            period_id,
            " | ",
            period_start.strftime(
                "%Y-%m-%d"
            ),
            " to ",
            period_end.strftime(
                "%Y-%m-%d"
            ),
            " | acquisitions = ",
            acquisition_count,
            sep=""
        )


        # ---------------------------------------------------------------------
        # No observations
        # ---------------------------------------------------------------------

        if acquisition_count == 0:

            print(
                "    No acquisition in this period."
            )


            biweekly_records.append(
                record
            )


            continue


        # ---------------------------------------------------------------------
        # Output filename
        # ---------------------------------------------------------------------

        output_file = (

            sensor_root /
            group /
            period /
            (
                f"{site_id}_"
                f"{period_id}_"
                f"{period_start.strftime('%Y-%m-%d')}_"
                f"{period_end.strftime('%Y-%m-%d')}_"
                f"biweekly_"
                f"{sensor}.tif"
            )

        )


        # ---------------------------------------------------------------------
        # Build/reuse
        # ---------------------------------------------------------------------

        result = (
            build_or_reuse_biweekly(

                source_files=
                    source_files,

                output_file=
                    output_file,

                band_names=
                    band_names,

            )
        )


        record.update(
            result
        )


        if (
            result.get(
                "composite_created",
                0
            )
            ==
            1
        ):

            record[
                "output_file"
            ] = (
                str(
                    output_file
                )
            )


        # ---------------------------------------------------------------------
        # Quality label and improvement
        # ---------------------------------------------------------------------

        if pd.notna(
            record.get(
                "valid_pixel_fraction"
            )
        ):

            current_quality = (
                record[
                    "valid_pixel_fraction"
                ]
            )


            record[
                "quality_label"
            ] = (
                quality_label(
                    current_quality
                )
            )


            if pd.notna(
                source_mean_valid_fraction
            ):

                record[
                    "improvement_vs_mean_source"
                ] = (

                    current_quality
                    -
                    source_mean_valid_fraction

                )


            if pd.notna(
                source_best_valid_fraction
            ):

                record[
                    "improvement_vs_best_source"
                ] = (

                    current_quality
                    -
                    source_best_valid_fraction

                )


            print(
                "    Valid pixels:",
                round(
                    current_quality
                    *
                    100,
                    2,
                ),
                "%"
            )


            print(
                "    Quality:",
                record[
                    "quality_label"
                ]
            )


            print(
                "    Action:",
                record[
                    "build_action"
                ]
            )


        biweekly_records.append(
            record
        )


# =============================================================================
# 27. Build detailed quality dataset
# =============================================================================

biweekly_quality = pd.DataFrame(
    biweekly_records
)


# =============================================================================
# 28. Quality thresholds
# =============================================================================

for threshold in [

    0.40,

    0.50,

    0.60,

    0.80,

    0.90,

]:

    threshold_pct = int(
        threshold
        *
        100
    )


    biweekly_quality[
        f"quality_ge_{threshold_pct}pct"
    ] = (
        biweekly_quality[
            "valid_pixel_fraction"
        ]
        >=
        threshold
    ).astype(int)


biweekly_quality[
    "quality_100pct"
] = (
    biweekly_quality[
        "valid_pixel_fraction"
    ]
    >=
    0.999999
).astype(int)


# =============================================================================
# 29. Save image-level quality CSV
# =============================================================================

biweekly_quality.to_csv(
    QUALITY_CSV_FILE,
    index=False,
)


print(
    "\nBiweekly image quality CSV:"
)


print(
    QUALITY_CSV_FILE
)


# =============================================================================
# 30. Build/reuse summary
# =============================================================================

build_action_summary = (
    biweekly_quality[
        "build_action"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "build_action"
    )
    .reset_index(
        name=
            "image_count"
    )
)


build_action_summary.to_csv(
    BUILD_ACTION_FILE,
    index=False,
)


print(
    "\nBuild/reuse summary:"
)


print(
    build_action_summary.to_string(
        index=False
    )
)


# =============================================================================
# 31. Overall quality summary
# =============================================================================

biweekly_summary = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        expected_site_periods=(
            "period_id",
            "count",
        ),

        periods_with_data=(
            "has_data",
            "sum",
        ),

        periods_with_multiple_acquisitions=(
            "multiple_acquisitions",
            "sum",
        ),

        total_acquisitions=(
            "acquisition_count",
            "sum",
        ),

        mean_acquisitions_per_period=(
            "acquisition_count",
            "mean",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        periods_ge_40pct_valid=(
            "quality_ge_40pct",
            "sum",
        ),

        periods_ge_50pct_valid=(
            "quality_ge_50pct",
            "sum",
        ),

        periods_ge_60pct_valid=(
            "quality_ge_60pct",
            "sum",
        ),

        periods_ge_80pct_valid=(
            "quality_ge_80pct",
            "sum",
        ),

        periods_ge_90pct_valid=(
            "quality_ge_90pct",
            "sum",
        ),

        mean_improvement_vs_mean_source=(
            "improvement_vs_mean_source",
            "mean",
        ),

        mean_improvement_vs_best_source=(
            "improvement_vs_best_source",
            "mean",
        ),

    )
)


biweekly_summary[
    "periods_without_data"
] = (

    biweekly_summary[
        "expected_site_periods"
    ]
    -
    biweekly_summary[
        "periods_with_data"
    ]

)


biweekly_summary[
    "percent_periods_with_data"
] = (

    biweekly_summary[
        "periods_with_data"
    ]
    /
    biweekly_summary[
        "expected_site_periods"
    ]
    *
    100

)


# =============================================================================
# 32. Threshold percentages
# =============================================================================

for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    biweekly_summary[
        f"percent_available_periods_ge_{threshold}pct"
    ] = np.where(

        biweekly_summary[
            "periods_with_data"
        ]
        >
        0,

        biweekly_summary[
            f"periods_ge_{threshold}pct_valid"
        ]
        /
        biweekly_summary[
            "periods_with_data"
        ]
        *
        100,

        np.nan,

    )


# =============================================================================
# 33. Convert fractions to percentages
# =============================================================================

for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    biweekly_summary[
        target_column
    ] = (
        biweekly_summary[
            source_column
        ]
        *
        100
    )


biweekly_summary[
    "mean_improvement_vs_mean_source_percentage_points"
] = (
    biweekly_summary[
        "mean_improvement_vs_mean_source"
    ]
    *
    100
)


biweekly_summary[
    "mean_improvement_vs_best_source_percentage_points"
] = (
    biweekly_summary[
        "mean_improvement_vs_best_source"
    ]
    *
    100
)


biweekly_summary.to_csv(
    SUMMARY_CSV_FILE,
    index=False,
)


# =============================================================================
# 34. PERIOD DIMENSION
#
# For each P01...P10, across all sites:
#
# How many treatment/control images are >=80% valid?
# =============================================================================

period_dimension = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
            "period_number",
            "period_id",
            "period_start",
            "period_end",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        images_with_data=(
            "has_data",
            "sum",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        images_ge_40pct=(
            "quality_ge_40pct",
            "sum",
        ),

        images_ge_50pct=(
            "quality_ge_50pct",
            "sum",
        ),

        images_ge_60pct=(
            "quality_ge_60pct",
            "sum",
        ),

        images_ge_80pct=(
            "quality_ge_80pct",
            "sum",
        ),

        images_ge_90pct=(
            "quality_ge_90pct",
            "sum",
        ),

    )
)


for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    period_dimension[
        f"percent_images_ge_{threshold}pct"
    ] = np.where(

        period_dimension[
            "images_with_data"
        ]
        >
        0,

        period_dimension[
            f"images_ge_{threshold}pct"
        ]
        /
        period_dimension[
            "images_with_data"
        ]
        *
        100,

        np.nan,

    )


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    period_dimension[
        target_column
    ] = (
        period_dimension[
            source_column
        ]
        *
        100
    )


period_dimension.to_csv(
    PERIOD_DIMENSION_FILE,
    index=False,
)


# =============================================================================
# 35. SITE DIMENSION
#
# For each individual treatment/counterfactual site:
#
# Across its 10 BEFORE or 10 AFTER periods,
# what percentage of images are >=80% valid?
# =============================================================================

site_dimension = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
            "site_id",
        ],
        as_index=False,
    )
    .agg(

        expected_periods=(
            "period_id",
            "count",
        ),

        periods_with_data=(
            "has_data",
            "sum",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        images_ge_40pct=(
            "quality_ge_40pct",
            "sum",
        ),

        images_ge_50pct=(
            "quality_ge_50pct",
            "sum",
        ),

        images_ge_60pct=(
            "quality_ge_60pct",
            "sum",
        ),

        images_ge_80pct=(
            "quality_ge_80pct",
            "sum",
        ),

        images_ge_90pct=(
            "quality_ge_90pct",
            "sum",
        ),

    )
)


site_dimension[
    "periods_without_data"
] = (

    site_dimension[
        "expected_periods"
    ]
    -
    site_dimension[
        "periods_with_data"
    ]

)


for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    site_dimension[
        f"percent_images_ge_{threshold}pct"
    ] = np.where(

        site_dimension[
            "periods_with_data"
        ]
        >
        0,

        site_dimension[
            f"images_ge_{threshold}pct"
        ]
        /
        site_dimension[
            "periods_with_data"
        ]
        *
        100,

        np.nan,

    )


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    site_dimension[
        target_column
    ] = (
        site_dimension[
            source_column
        ]
        *
        100
    )


site_dimension.to_csv(
    SITE_DIMENSION_FILE,
    index=False,
)


# =============================================================================
# 36. Quality definitions
# =============================================================================

quality_definitions = pd.DataFrame(
    {

        "variable": [

            "valid_pixel_fraction",

            "valid_pixel_fraction_any_band",

            "valid_pixel_fraction_all_bands",

            "acquisition_count",

            "source_mean_valid_pixel_fraction",

            "source_best_valid_pixel_fraction",

            "improvement_vs_mean_source",

            "improvement_vs_best_source",

            "build_action",

        ],


        "meaning": [

            (
                "PRIMARY quality metric. Fraction of spatial pixels where "
                "at least one output band has a finite value."
            ),

            (
                "Explicit any-band definition; equivalent to the primary "
                "valid_pixel_fraction."
            ),

            (
                "Strict diagnostic requiring every output band to be valid "
                "for the spatial pixel."
            ),

            (
                "Number of actual Notebook 10 acquisition dates contributing "
                "to the 14-day composite."
            ),

            (
                "Mean primary valid-pixel fraction across source acquisitions."
            ),

            (
                "Highest primary valid-pixel fraction among source "
                "acquisitions."
            ),

            (
                "Biweekly valid fraction minus average source-image quality."
            ),

            (
                "Biweekly valid fraction minus best source-image quality."
            ),

            (
                "created = new TIFF; skipped_existing = valid TIFF already "
                "existed; missing = zero acquisitions; failed = error."
            ),

        ],

    }
)


# =============================================================================
# 37. Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        QUALITY_EXCEL_FILE,
        engine=
            "openpyxl",
    ) as writer:

        biweekly_quality.to_excel(
            writer,
            sheet_name=
                "image_quality",
            index=False,
        )


        biweekly_summary.to_excel(
            writer,
            sheet_name=
                "summary",
            index=False,
        )


        period_dimension.to_excel(
            writer,
            sheet_name=
                "period_dimension",
            index=False,
        )


        site_dimension.to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        period_definitions.to_excel(
            writer,
            sheet_name=
                "period_definitions",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "quality_definitions",
            index=False,
        )


        build_action_summary.to_excel(
            writer,
            sheet_name=
                "build_actions",
            index=False,
        )


    print(
        "\nExcel workbook saved:"
    )


    print(
        QUALITY_EXCEL_FILE
    )


except ModuleNotFoundError:

    print(
        "\nopenpyxl is not installed."
    )


    print(
        "All CSV outputs were still generated."
    )


    print(
        "Install with:"
    )


    print(
        "%pip install openpyxl"
    )


# =============================================================================
# 38. Folder summary
# =============================================================================

folder_records = []


for sensor_name, sensor_root in [

    (
        "sentinel1",
        S1_BIWEEKLY_DIR,
    ),

    (
        "sentinel2",
        S2_BIWEEKLY_DIR,
    ),

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (
                sensor_root /
                group /
                period
            )


            folder_records.append(
                {

                    "sensor":
                        sensor_name,

                    "group":
                        group,

                    "period":
                        period,

                    "folder":
                        str(
                            folder
                        ),

                    "biweekly_tiff_count":
                        len(
                            list(
                                folder.glob(
                                    "*.tif"
                                )
                            )
                        ),

                }
            )


folder_summary = pd.DataFrame(
    folder_records
)


folder_summary.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False,
)


# =============================================================================
# 39. Site coverage check
# =============================================================================

coverage_check = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
        ],
        as_index=False,
    )
    .agg(

        unique_sites=(
            "site_id",
            "nunique",
        ),

        total_site_period_rows=(
            "period_id",
            "count",
        ),

        periods_with_data=(
            "has_data",
            "sum",
        ),

        composites_created=(
            "composite_created",
            "sum",
        ),

    )
)


print(
    "\n"
    + "=" * 100
)


print(
    "SITE COVERAGE CHECK"
)


print(
    "=" * 100
)


print(
    coverage_check.to_string(
        index=False
    )
)


# =============================================================================
# 40. Main quality summary
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "BIWEEKLY IMAGE QUALITY SUMMARY"
)


print(
    "=" * 100
)


columns_to_show = [

    "sensor",

    "group",

    "period",

    "number_of_sites",

    "expected_site_periods",

    "periods_with_data",

    "periods_with_multiple_acquisitions",

    "mean_acquisitions_per_period",

    "mean_valid_pixel_percentage",

    "median_valid_pixel_percentage",

    "periods_ge_80pct_valid",

    "percent_available_periods_ge_80pct",

]


print(
    biweekly_summary[
        columns_to_show
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 41. Sentinel-2 period dimension
# =============================================================================

s2_period = (
    period_dimension
    .loc[
        period_dimension[
            "sensor"
        ]
        ==
        "sentinel2"
    ]
    .copy()
)


print(
    "\n"
    + "=" * 100
)


print(
    "SENTINEL-2 PERIOD × SITE QUALITY"
)


print(
    "=" * 100
)


print(
    s2_period[
        [
            "group",
            "period",
            "period_id",
            "number_of_sites",
            "images_with_data",
            "mean_valid_pixel_percentage",
            "median_valid_pixel_percentage",
            "images_ge_80pct",
            "percent_images_ge_80pct",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 42. Validate period structure
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORAL VALIDATION"
)


print(
    "=" * 100
)


print(
    "\nBefore periods:"
)


print(
    len(
        before_definition
    )
)


print(
    "\nAfter periods:"
)


print(
    len(
        after_definition
    )
)


print(
    "\nFirst before period:"
)


print(
    before_definition.iloc[
        0
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nLast before period:"
)


print(
    before_definition.iloc[
        -1
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nFirst after period:"
)


print(
    after_definition.iloc[
        0
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nLast after period:"
)


print(
    after_definition.iloc[
        -1
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


# =============================================================================
# 43. Final summary
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "NOTEBOOK 12 COMPLETE"
)


print(
    "=" * 100
)


print(
    "\nSource acquisition-level data:"
)


print(
    DAILY_DIR
)


print(
    "\nBiweekly output:"
)


print(
    BIWEEKLY_DIR
)


print(
    "\nTemporal structure:"
)


print(
    "10 BEFORE × 14 days"
)


print(
    "10 AFTER  × 14 days"
)


print(
    "\nBefore:"
)


print(
    "2024-05-10 through 2024-09-26"
)


print(
    "\nAfter:"
)


print(
    "2024-09-27 through 2025-02-13"
)


print(
    "\nBiweekly quality CSV:"
)


print(
    QUALITY_CSV_FILE
)


print(
    "\nBiweekly quality Excel:"
)


print(
    QUALITY_EXCEL_FILE
)


print(
    "\nSummary:"
)


print(
    SUMMARY_CSV_FILE
)


print(
    "\nPeriod dimension:"
)


print(
    PERIOD_DIMENSION_FILE
)


print(
    "\nSite dimension:"
)


print(
    SITE_DIMENSION_FILE
)


print(
    "\nPeriod definitions:"
)


print(
    PERIOD_DEFINITION_FILE
)


print(
    "\nBuild/reuse summary:"
)


print(
    BUILD_ACTION_FILE
)


print(
    "\nSkip existing biweekly TIFFs:"
)


print(
    SKIP_EXISTING_BIWEEKLY
)


if RUN_TEST_ONLY:

    print(
        "\nTEST MODE ACTIVE."
    )


else:

    print(
        "\nFULL DATASET MODE ACTIVE."
    )


    print(
        "All treatment and counterfactual sites represented "
        "in Notebook 10 are processed."
    )


    print(
        "\nIf Notebook 12 is interrupted, rerun it."
    )


    print(
        "Existing valid biweekly TIFFs will not be rebuilt."
    )


print(
    "\nNo Earth Engine download is performed by Notebook 12."
)


print(
    "\nNotebook completed successfully."
)